# Export CSV to SQL - Complete Workflow

This notebook demonstrates how to transform CSV data and export it as SQL INSERT statements.

The workflow:
1. Load CSV data
2. Apply transformations (lookups, computed columns, etc.)
3. Convert to SQL INSERT statements with batching


In [5]:
from pathlib import Path
from datetime import datetime

import pandas as pd

from db import Base, engine, get_session
from models import Customer
from transformer import TransformationConfig
from sql_writer import csv_to_sql, dataframe_to_batched_inserts, write_dataframe_as_batched_inserts

# Create all tables
Base.metadata.create_all(bind=engine)
print("Database tables ready.")

Database tables ready.


## Method 1: Direct CSV to SQL Conversion

Use the generic `csv_to_sql()` function for simple conversions without transformations.

In [ ]:
# Convert CSV directly to SQL with batched INSERT statements
csv_to_sql(
    csv_path="current-data/customers_export.csv",
    table_name="customers",
    output_path="clean-data/customers.sql",
    batch_size=1000  # Group 1000 rows per INSERT statement
)

## Method 2: CSV to SQL with Specific Columns

Export only specific columns from the CSV.

In [ ]:
# Export only specific columns
csv_to_sql(
    csv_path="current-data/customers_export.csv",
    table_name="customers",
    output_path="sql_output/customers_partial.sql",
    columns=["id", "customer_name", "email"],
    batch_size=500
)

## Method 3: Transform CSV, Then Export to SQL

Apply transformations to CSV data before converting to SQL.

In [2]:
# Load CSV
input_df = pd.read_csv("current-data/customers_export.csv")
print(f"Loaded {len(input_df)} rows from CSV")
input_df.head()

Loaded 2470 rows from CSV


,id,tin_number,customer_name,customer_vat_number,email,region,city,business_type,primary_contact,secondary_contact,created_at,updated_at
0,1099,0048890966,Nullname,nulvat,nulemil,nullregion,nulcity,nulbusiness,nulprimary,nulsecondary,NaN,NaN
1,1101,0039855136,DAS ANA Imp AND exp/Adane AND His Family Tradi...,nulvat,nulemil,nullregion,nulcity,nulbusiness,nulprimary,nulsecondary,NaN,NaN
2,1102,0045710281,KAFFA FOREST BEE PRODUCT,nulvat,nulemil,nullregion,nulcity,nulbusiness,nulprimary,nulsecondary,NaN,NaN
3,1103,000000000000000,AAAAAAAAAAAASELEAAA,nulvat,nulemil,nullregion,nulcity,nulbusiness,nulprimary,nulsecondary,NaN,NaN
4,1104,0005706123,Aarti Steel PLC,nulvat,nulemil,nullregion,nulcity,nulbusiness,nulprimary,nulsecondary,NaN,NaN


In [ ]:
# Define transformations
def full_name_func(row):
    name = row.get("customer_name") or ""
    return str(name).strip()

config = TransformationConfig(
    replace_lookups=[
        {
            "source_column": "customer_tin",
            "model": "models:Customer",
            "lookup_field": "tin_number",
            "return_field": "id",
            "new_column_name": "customer_id",
        }
    ],
    remove_columns=["temp_column", "unused_flag"],
    computed_columns=[
        {
            "new_column": "full_name",
            "function": full_name_func,
        }
    ],
    required_columns={
        "created_at": datetime.utcnow,
    },
)

# Apply transformations
with get_session() as session:
    transformed_df = config.apply(input_df.copy(), session)

print(f"Transformed {len(transformed_df)} rows")
transformed_df.head()

In [ ]:
# Convert transformed DataFrame to SQL with batched INSERT statements
write_dataframe_as_batched_inserts(
    df=transformed_df,
    table_name="customers",
    output_path="clean-data/customers_transformed.sql",
    batch_size=1000
)

print("Transformed data exported to SQL!")

## Method 4: Generate SQL Statements in Memory

Get the SQL statements as a list without writing to file.

In [3]:
# Generate batched INSERT statements
statements = dataframe_to_batched_inserts(
    df=transformed_df,
    table_name="customers",
    batch_size=1000
)

print(f"Generated {len(statements)} batched INSERT statements")
print("\nFirst statement preview:")
print(statements[0][:500] + "..." if len(statements[0]) > 500 else statements[0])

NameError: name 'transformed_df' is not defined

## Complete ETL Pipeline

Putting it all together: Load → Transform → Export to SQL

In [ ]:
# 1. Load CSV
df = pd.read_csv("current-data/customers_export.csv")

# 2. Apply transformations (if needed)
with get_session() as session:
    transformed = config.apply(df.copy(), session)

# 3. Export to SQL
write_dataframe_as_batched_inserts(
    df=transformed,
    table_name="customers",
    output_path="clean-data/customers_final.sql",
    batch_size=1000
)

print("✓ ETL pipeline complete!")